# AI Ride Call — merged six-stage pipeline

Uses the applied working-project source plus missing resources from the input ZIP. Run in a fresh Kaggle GPU session with Internet enabled and the required secrets. Historical patch notebooks are not needed.

See README.md and docs/VALIDATION.md for exact validation scopes and inherited test findings.


In [ ]:
from pathlib import Path
import sys

# Set an exact repository root here if more than one copy is attached.
REPO_OVERRIDE = ""
if REPO_OVERRIDE:
    REPO = Path(REPO_OVERRIDE).resolve()
else:
    candidates = [Path.cwd()] if (Path.cwd() / "pipeline.py").is_file() else []
    if not candidates:
        candidates = [p.parent for p in Path("/kaggle/input").rglob("pipeline.py")
                      if (p.parent / "docs/SOURCE_MAP.json").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Attach exactly one complete repository, or set REPO_OVERRIDE.")
    REPO = candidates[0]
assert (REPO / "requirements.txt").is_file()
print("Repository:", REPO)


## Install dependencies once

Run before importing model packages. If model packages were already imported, restart the kernel after installation and rerun the repository-location cell. The pinned Hugging Face stack can differ from unrelated preinstalled Kaggle packages. Use this project in its own fresh session.

In [ ]:
import subprocess
if globals().get("bridge") is not None or globals().get("SCHEDULER") is not None:
    raise RuntimeError("Use a fresh kernel before installing dependencies.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")])
print("Dependencies installed. Continue in this fresh kernel.")


## Start the complete ordered pipeline

The stages stop on the first error and retain the error output. A failed partial startup requires a fresh session. Keep the notebook running while testing the shared HTTPS website.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("movio_repository_pipeline", REPO / "pipeline.py")
movio_pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = movio_pipeline
spec.loader.exec_module(movio_pipeline)
movio_pipeline.run_pipeline(globals())


## Required browser acceptance

Open the printed shared HTTPS URL. Use separate browser profiles for customer and driver. Test a matching vehicle, a future booking within the dispatch window, final confirmation, saved booking details, driver acceptance, OTP and cancellation. Check all three languages, weather popup/transcript/playback, microphone permission and Travel Companion location permission. Confirm that selected locations remain unchanged by companion lookups.

Offline component fixtures are in `scripts/run_component_checks.py`. They do not establish real speech quality or end-to-end booking success. Historical benchmark outputs were not copied into this startup notebook.